# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connection, lane constants, the w03 slice

Same lane, same contract, same slice as w03: **features measured in `month=2026-03`, label
measured in `month=2026-04`, decision moment 2026-03-31**. The rule I build this week has to be
scored on exactly the data the model will be scored on next week, so nothing here is re-chosen —
the constants below are copied from the w03 data contract and the slice is rebuilt from the same
two queries.

Two things are new this week: `max_day_imp` (the biggest single day of March, used to spot pages
whose "slide" is really one spike leaving the window) and the `dim_content` update dates, which
signal check 1a puts on trial.

In [1]:
import importlib.util, subprocess, sys

for pkg in ("duckdb", "huggingface_hub"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, pathlib
import duckdb, numpy as np, pandas as pd

pd.set_option("display.width", 150)
SEED = 42  # same seed as w03, so the grouped split below is the same split


def load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "environment variable"
    for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        env_file = parent / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip().strip("\"'"), "local .env (gitignored)"
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab Secret"
    except Exception:
        pass
    raise RuntimeError("No HF_TOKEN found. Set a Colab Secret named HF_TOKEN (read token).")


HF_TOKEN, token_source = load_hf_token()
print(f"HF read token loaded from: {token_source}")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"


def fact(month):
    """One month partition of fact_content_daily_performance."""
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')"


# --- lane constants: copied unchanged from the w03 data contract -------------
FEATURE_MONTH = "2026-03"          # features are measured here
LABEL_MONTH   = "2026-04"          # strictly after the decision moment
DECISION_DATE = "2026-03-31"       # the moment the editor scores the queue
MID_MONTH     = "2026-03-17"       # splits the feature month into two halves
MIN_HISTORY   = "2026-01-01"       # client must have GSC history starting on/before this
MIN_IMP_MAR   = 100                # a page needs measurable March search demand
DECLINE_DROP  = 0.20               # "declining" = April impressions >20% below March
TOP_K         = 50                 # editorial capacity: the queue is 50 pages long

receipts = {"lane": "bleed_tracker", "seed": SEED, "top_k": TOP_K,
            "feature_month": FEATURE_MONTH, "label_month": LABEL_MONTH}


def precision_at_k(df, score_col, label_col="label_declining", k=TOP_K):
    """Of the top k rows this score ranks highest, how many were actually declining?"""
    return float(df.nlargest(k, score_col)[label_col].mean())


print(f"decision moment {DECISION_DATE} | features from {FEATURE_MONTH} | label from {LABEL_MONTH}")

HF read token loaded from: local .env (gitignored)
decision moment 2026-03-31 | features from 2026-03 | label from 2026-04


### 0b. Rebuild the contract slice

One row = one page-month decision row, exactly as w03 defined it: `gsc_data_available IS TRUE`,
clients with GSC history from before 2026-01-01, pages with at least 100 March impressions. The
last cell of this section checks the rebuilt slice against w03's committed receipts — if the row
count or the base rate had drifted, every number below would be comparing against a different
week's work.

In [2]:
march = con.sql(f"""
    WITH m AS (
        SELECT f.client_hash_id,
               f.content_hash_id,
               SUM(f.gsc_impressions)                                          AS imp_mar,
               SUM(f.gsc_clicks)                                               AS clk_mar,
               COUNT(*) FILTER (f.gsc_impressions > 0)                         AS active_days,
               SUM(f.gsc_sum_position)                                         AS sum_pos_mar,
               MAX(f.gsc_impressions)                                          AS max_day_imp,
               SUM(f.gsc_impressions) FILTER (f.report_date >= DATE '{MID_MONTH}') AS imp_late,
               SUM(f.gsc_impressions) FILTER (f.report_date <  DATE '{MID_MONTH}') AS imp_early
        FROM {fact(FEATURE_MONTH)} f
        JOIN {DIM_CLIENTS} c USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
          AND c.gsc_data_start <= DATE '{MIN_HISTORY}'
        GROUP BY 1, 2
        HAVING SUM(f.gsc_impressions) >= {MIN_IMP_MAR}
    )
    SELECT m.*,
           d.content_updated_date          -- section 1a puts this column on trial
    FROM m LEFT JOIN {DIM_CONTENT} d USING (content_hash_id)
""").df()

april = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr
    FROM {fact(LABEL_MONTH)}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
""").df()

assert not march.content_hash_id.duplicated().any(), "content_hash_id is not unique in-month"
assert not april.content_hash_id.duplicated().any(), "label key is not unique"

frame = march.merge(april, on="content_hash_id", how="left")
assert len(frame) == len(march), "the label join changed the row count"
frame["imp_apr"] = frame.imp_apr.fillna(0)

# --- the five contract features, rebuilt exactly as in w03 -------------------
frame["pos_mar"] = frame.sum_pos_mar / frame.imp_mar
frame["ctr_mar"] = frame.clk_mar / frame.imp_mar
frame["momentum_in_month"] = frame.imp_late / frame.imp_early.replace(0, np.nan)
frame["has_first_half_traffic"] = frame.momentum_in_month.notna().astype(int)
frame["momentum_in_month"] = frame.momentum_in_month.fillna(1.0)   # 1.0 = "flat", flagged above
frame["log_imp_mar"] = np.log1p(frame.imp_mar)
frame["spike_share"] = frame.max_day_imp / frame.imp_mar           # new this week: spike detector

# --- the label (the only column measured after the decision moment) ----------
frame["label_declining"] = (frame.imp_apr < (1 - DECLINE_DROP) * frame.imp_mar).astype(int)

CONTRACT_FEATURES = ["log_imp_mar", "active_days", "pos_mar", "ctr_mar", "momentum_in_month"]

print(f"slice: {len(frame):,} page-month rows | {frame.client_hash_id.nunique()} clients")
print(f"base rate (share declining next month): {frame.label_declining.mean():.2%}")

# The slice must be the one w03 signed off on, not a lookalike.
w03 = pathlib.Path("../outputs/w03_data_contract_receipts.json")
if w03.exists():
    prev = json.loads(w03.read_text(encoding="utf-8"))
    assert len(frame) == prev["frame_rows"], "slice size drifted from the w03 contract"
    assert round(float(frame.label_declining.mean()), 4) == prev["label_base_rate"], "base rate drifted"
    print(f"cross-check vs work/outputs/w03_data_contract_receipts.json: rows and base rate match "
          f"({prev['frame_rows']:,} rows, {prev['label_base_rate']:.2%})")
else:
    print("w03 receipts not found next to this notebook — skipping the cross-check.")

receipts["slice_rows"] = int(len(frame))
receipts["slice_clients"] = int(frame.client_hash_id.nunique())
receipts["slice_base_rate"] = round(float(frame.label_declining.mean()), 4)

slice: 85,453 page-month rows | 27 clients
base rate (share declining next month): 53.78%
cross-check vs work/outputs/w03_data_contract_receipts.json: rows and base rate match (85,453 rows, 53.78%)


### 0c. The split, fixed before anything is measured

Every audit table in section 1 is computed on the **training clients only**, and the queue is
scored on the **held-out clients** in section 2. The split is the same one w03 used —
`GroupShuffleSplit(test_size=0.3, random_state=42)` grouped by client — so this week's baseline
number and next week's model number are read off the same held-out rows.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
train_idx, test_idx = next(splitter.split(frame, groups=frame.client_hash_id))
frame["split"] = "train"
frame.loc[frame.index[test_idx], "split"] = "test"

train, test = frame[frame.split == "train"], frame[frame.split == "test"]
print(f"train {len(train):,} rows / {train.client_hash_id.nunique()} clients — every audit table below")
print(f"test  {len(test):,} rows / {test.client_hash_id.nunique()} clients — sealed until section 2")
print(f"\ntrain base rate {train.label_declining.mean():.2%} | test base rate {test.label_declining.mean():.2%}")
print("(the two differ by client mix — the test base rate is the number the queue has to beat)")

receipts["train_rows"], receipts["test_rows"] = int(len(train)), int(len(test))
receipts["test_clients"] = int(test.client_hash_id.nunique())
receipts["test_base_rate"] = round(float(test.label_declining.mean()), 4)

train 59,682 rows / 18 clients — every audit table below
test  25,771 rows / 9 clients — sealed until section 2

train base rate 55.69% | test base rate 49.36%
(the two differ by client mix — the test base rate is the number the queue has to beat)


## 1. My rule and its reason codes

**The rule idea, before any code:** a page belongs at the top of the editor's queue when it
*slid inside the month it was measured*, when that slide is happening on a page big enough for
the loss to matter, and when it is under-earning clicks for the rank it already holds.

That idea leans on three signals, so all three get checked **first**, on training clients only,
with a bucket table and a printed `n` next to every rate. Two of them sit behind rules FlyRank
already ships:

| # | Signal | The real FlyRank flag behind it | Verdict |
|---|---|---|---|
| 1a | staleness (`days_since_update`) | the refresh flags — `stale_visible_page`: not updated in 180+ days and 500+ impressions | **FALSE** (as a usable signal in this slice) |
| 1b | CTR for the position held | the CTR-fix logic — `low_ctr_visible_page` / `needs_ctr_fix`: 500+ impressions, position 1–20, CTR < 0.5% | **MIXED** |
| 1c | the within-month slide | none — this one is mine | **CONFIRMED** |

Verdict words are the four from the audit protocol: **CONFIRMED** (holds in the direction
claimed), **OPPOSITE** (holds, backwards), **MIXED** (holds in part), **FALSE** (does not hold
here). A negative verdict is not a wasted check.

### 1a. Signal check — staleness, the signal behind the refresh flags

**What the flag assumes.** FlyRank's refresh flag `stale_visible_page` fires when a page has not
been updated in 180+ days and still pulls 500+ impressions. The assumption underneath it: *pages
go stale, and stale visible pages are the ones quietly bleeding.* If that holds here, staleness
belongs in my score.

**What I test.** The warehouse's `dim_content.content_updated_date` is the only update signal in
the release. I bucket the training pages by how many days before the decision moment
(2026-03-31) they were last updated, and read the decline rate in each bucket with its `n`.

**The first thing the query has to answer is whether the column is even knowable at the decision
moment** — `dim_content` is a snapshot taken when the release was built (2026-07-03), not a
history table, so an update that happened in May is recorded as if it were the page's state all
along.

In [4]:
dec = pd.Timestamp(DECISION_DATE)
train_s = train.copy()
train_s["days_since_update"] = (dec - pd.to_datetime(train_s.content_updated_date)).dt.days

after = (train_s.days_since_update < 0)
print(f"pages whose recorded last update is AFTER the decision moment: {after.sum():,} of "
      f"{len(train_s):,} ({after.mean():.1%})")
print("-> for those, the value the editor would have seen on 2026-03-31 is not in this release at all.\n")

known = train_s[~after].copy()
known["stale_bucket"] = pd.cut(known.days_since_update, [-1, 89, 179, 364, 10**6],
                               labels=["0-89d", "90-179d", "180-364d", "365d+"])
tbl_1a = (known.groupby("stale_bucket", observed=True)
                .agg(n=("label_declining", "size"),
                     decline_rate=("label_declining", "mean"),
                     median_imp_mar=("imp_mar", "median")))
print("staleness at the decision moment vs decline next month (training clients, knowable dates only)")
print(tbl_1a.round(4).to_string())

flag_pool = ((known.days_since_update >= 180) & (known.imp_mar >= 500)).sum()
print(f"\npages the shipped flag would fire on (>=180 days stale AND >=500 impressions): {flag_pool}")
print(f"testable pool: {len(known):,} of {len(train_s):,} training pages "
      f"({len(known) / len(train_s):.1%}) — and that pool is defined by NOT having been touched "
      f"after March, which is itself information from after the decision moment.")

receipts["s1a_updated_after_decision_share"] = round(float(after.mean()), 4)
receipts["s1a_testable_pages"] = int(len(known))
receipts["s1a_flag_pool"] = int(flag_pool)
receipts["s1a_verdict"] = "FALSE"

pages whose recorded last update is AFTER the decision moment: 51,377 of 59,682 (86.1%)
-> for those, the value the editor would have seen on 2026-03-31 is not in this release at all.

staleness at the decision moment vs decline next month (training clients, knowable dates only)
                 n  decline_rate  median_imp_mar
stale_bucket                                    
0-89d         8244        0.6087           785.0
90-179d         51        0.4706           153.0
180-364d        10        1.0000           224.5

pages the shipped flag would fire on (>=180 days stale AND >=500 impressions): 4
testable pool: 8,305 of 59,682 training pages (13.9%) — and that pool is defined by NOT having been touched after March, which is itself information from after the decision moment.


**Verdict: FALSE** — not "staleness doesn't matter in SEO", but "this release cannot support a
staleness rule, and a rule built on it would be a rule built on the future."

Three reasons, in the order the numbers show them:

1. **The column is contaminated by design.** 86.1% of training pages carry a last-update date
   *after* 2026-03-31. `dim_content` is a current-state snapshot, so for those pages the value I
   would score on is a fact from April–July — the wrong side of the decision moment.
2. **The part I can test is a survivor sample.** The 8,305 pages with a knowable date are exactly
   the pages nobody touched in the four months after the decision moment. Selecting them uses
   post-decision information too, so even their honest-looking table is not evidence about the
   population I would score.
3. **Inside that sample the direction doesn't hold anyway.** Freshly-updated pages (0–89 days,
   n=8,244) decline at 60.9% while the 90–179 day bucket (n=51) declines at 47.1%. The
   180–364 day bucket is 10 pages — a 100% rate on ten pages is a coin landing heads three times.
   And the shipped flag's own condition (180+ days stale *and* 500+ impressions) fires on **4
   pages** in 59,682. A flag that can't fire can't rank a queue.

**What this changes:** staleness is out of my score entirely — no `days_since_update` factor, no
"stale" reason code. To test the refresh flag properly the release would need an as-of-date
update history (a dated `content_updated` event, or a slowly-changing dimension I could query as
of 2026-03-31), and this build ships neither. That is a finding I can hand back, not a gap I can
patch in a notebook.

### 1b. Signal check — CTR for the position it holds, the signal behind the CTR-fix flag

**What the flag assumes.** `low_ctr_visible_page` / `needs_ctr_fix` fires on a page with 500+
impressions, an average position of 1–20, and CTR under 0.5%. The assumption: *a page that ranks
well but earns few clicks is underperforming, and it is worth a human's time.*

**What I test.** Comparing raw CTR across pages is unfair — a page at position 2 and a page at
position 18 live in different worlds. So I bucket by position band, then split each band at its
own **median CTR** (medians computed on training clients only) and read the decline rate on each
side, with `n`. Then I check the shipped 0.5% threshold on its own terms: how much of the
eligible pool does it actually fire on?

In [5]:
POS_BANDS = [0, 3, 10, 20, np.inf]
POS_NAMES = ["1-3", "4-10", "11-20", "21+"]

train_s["pos_band"] = pd.cut(train_s.pos_mar, POS_BANDS, labels=POS_NAMES)
BAND_MED_CTR = train_s.groupby("pos_band", observed=True)["ctr_mar"].median()   # learned on TRAIN only
print("median March CTR per position band (training clients — these become the rule's yardstick)")
print((BAND_MED_CTR * 100).round(4).rename("median_ctr_pct").to_string())

train_s["underclick"] = (train_s.ctr_mar < train_s.pos_band.map(BAND_MED_CTR).astype(float)).astype(int)
tbl_1b = (train_s.groupby(["pos_band", "underclick"], observed=True)
                 .agg(n=("label_declining", "size"), decline_rate=("label_declining", "mean")))
print("\nbelow-median CTR for its own position band (underclick=1) vs decline next month")
print(tbl_1b.round(4).to_string())

gaps = (tbl_1b.decline_rate.unstack()[1] - tbl_1b.decline_rate.unstack()[0]).dropna()
print("\ngap in decline rate (underclicking minus not), percentage points:")
print((gaps * 100).round(1).to_string())

# the shipped threshold, judged on its own terms
elig = train_s[(train_s.imp_mar >= 500) & (train_s.pos_mar >= 1) & (train_s.pos_mar <= 20)].copy()
elig["ctr_fix_flag"] = (elig.ctr_mar * 100 < 0.5).astype(int)
print(f"\nshipped-threshold pool (>=500 impressions, position 1-20): n={len(elig):,}")
print(f"the 0.5% CTR threshold fires on {int(elig.ctr_fix_flag.sum()):,} of them "
      f"({elig.ctr_fix_flag.mean():.1%})")
print(elig.groupby("ctr_fix_flag").agg(n=("label_declining", "size"),
                                       decline_rate=("label_declining", "mean")).round(4).to_string())

receipts["s1b_band_median_ctr_pct"] = {k: round(float(v) * 100, 4) for k, v in BAND_MED_CTR.items()}
receipts["s1b_underclick_gap_pp"] = {str(k): round(float(v) * 100, 1) for k, v in gaps.items()}
receipts["s1b_shipped_threshold_fire_rate"] = round(float(elig.ctr_fix_flag.mean()), 4)
receipts["s1b_verdict"] = "MIXED"

median March CTR per position band (training clients — these become the rule's yardstick)
pos_band
1-3      0.1775
4-10     0.1889
11-20    0.1165
21+      0.0000

below-median CTR for its own position band (underclick=1) vs decline next month
                         n  decline_rate
pos_band underclick                     
1-3      0            2464        0.5556
         1            2463        0.7479
4-10     0           12881        0.4472
         1           12878        0.6788
11-20    0            6321        0.4727
         1            6321        0.6209
21+      0           16354        0.5264

gap in decline rate (underclicking minus not), percentage points:
pos_band
1-3      19.2
4-10     23.2
11-20    14.8

shipped-threshold pool (>=500 impressions, position 1-20): n=28,224
the 0.5% CTR threshold fires on 23,276 of them (82.5%)
                  n  decline_rate
ctr_fix_flag                     
0              4948        0.3577
1             23276        0.5985


**Verdict: MIXED** — the flag's *direction* is confirmed; the flag's *threshold* is not usable in
this slice.

- **Direction — confirmed.** In every position band, pages earning below-median CTR for their
  band decline more often next month, and the gap is large: **+19.2pp** at positions 1–3 (74.8%
  vs 55.6%), **+23.2pp** at 4–10 (67.9% vs 44.7%), **+14.8pp** at 11–20 (62.1% vs 47.3%). "Ranks
  fine, earns few clicks" really does mark a page in trouble, not just a page with a boring title.
- **Threshold — not usable.** The shipped 0.5% cut fires on **82.5%** of its own eligible pool.
  Median CTR in this monthly warehouse slice is 0.12–0.19% depending on band, so a 0.5% line sits
  far above almost everything: it separates (59.9% vs 35.8% decline) but it cannot *rank*, and
  a queue is a ranking problem. The number was tuned on 90-day starter data where CTR is pooled
  over three months; monthly per-page CTR is a different scale.
- **Position 21+ has a median CTR of exactly 0.0000%**, so no page can fall below it — the
  underclick test simply never fires in that band. Worth knowing before the score uses it.

**What this changes:** my rule uses **relative** CTR (below the median of its own position band),
not the absolute 0.5% line — and it treats "underclicking" as a multiplier on top of the slide,
never as an entry ticket on its own.

### 1c. Signal check — the within-month slide (the signal my rule leans on hardest)

**What I claim.** `momentum_in_month` = March 17–31 impressions ÷ March 1–16 impressions. A page
whose second half ran well below its first half was *already sliding when the editor looked at
it*. This is not behind a FlyRank flag — it is the signal I am proposing — so it gets the same
test before I trust it with the top of the queue.

Both halves sit inside the feature month, so nothing here is knowable only in April. The two
halves are 15 and 16 days, so a perfectly flat page scores just under 1.0 (w03 measured the
median at 0.99); the tilt applies to every row equally, so it moves the scale, not the ranking.

In [6]:
train_s["mom_bucket"] = pd.cut(train_s.momentum_in_month, [-0.01, 0.5, 0.8, 1.2, 2.0, 10**9],
                               labels=["<0.5", "0.5-0.8", "0.8-1.2", "1.2-2.0", ">2.0"])
tbl_1c = (train_s.groupby("mom_bucket", observed=True)
                 .agg(n=("label_declining", "size"),
                      decline_rate=("label_declining", "mean"),
                      median_imp_mar=("imp_mar", "median")))
print("second half of March vs first half, and what happened in April (training clients)")
print(tbl_1c.round(4).to_string())
print(f"\ntrain base rate for reference: {train.label_declining.mean():.2%}")
print(f"pages with no first-half traffic (momentum undefined, filled 1.0 and flagged): "
      f"{int((1 - train.has_first_half_traffic).sum()):,}")

receipts["s1c_decline_by_momentum"] = {str(k): round(float(v), 4)
                                       for k, v in tbl_1c.decline_rate.items()}
receipts["s1c_verdict"] = "CONFIRMED"

second half of March vs first half, and what happened in April (training clients)
                n  decline_rate  median_imp_mar
mom_bucket                                     
<0.5         6906        0.8655           613.0
0.5-0.8     11694        0.7349          1004.0
0.8-1.2     18507        0.5471           973.0
1.2-2.0     13059        0.4235           930.0
>2.0         9516        0.3159           508.0

train base rate for reference: 55.69%
pages with no first-half traffic (momentum undefined, filled 1.0 and flagged): 1,141


**Verdict: CONFIRMED, and it is the strongest of the three** — decline rates fall monotonically
across the buckets: **86.6%** for pages whose second half was under half the first (n=6,906),
73.5% (n=11,694), 54.7%, 42.4%, down to **31.6%** for pages that more than doubled (n=9,516),
against a 55.7% training base rate. The spread is 55 percentage points, and it is monotone —
there is no threshold to hand-pick, the whole gradient is usable.

**One honest caveat I carry forward into section 4:** a page that collapsed in the second half of
March will *mechanically* tend to post a lower April total, because the label compares April to
the whole of March — including the healthy first half. So this signal is partly measuring
"a slide already under way continues", not "a healthy page is about to turn". That is legitimate
— it is entirely knowable on 2026-03-31 — but it means my precision is about **persistence**, and
I say so where I report the number.

**What this changes:** the slide becomes the rule's gate and its main gradient — not a yes/no
threshold but a severity, so a page at 0.05 outranks a page at 0.7.

### 1d. The rule, in three sentences

> **Only pages that were already sliding inside March enter the queue.** Among those, a page
> ranks higher the deeper its slide and the more impressions it stands to lose — and it is
> pushed further up if it also under-earns clicks for the position it holds, or sits in reach
> (positions 1–20) of an edit that could move it. Everything else is a `monitor`, not a task.

As code, with no fitted weights anywhere:

```text
severity   = clip(1 - momentum_in_month / 0.8, 0, 1)     # 0 = not sliding, 1 = second half collapsed
underclick = ctr_mar < median CTR of its position band   # 1b, relative not absolute
reachable  = 1 <= pos_mar <= 20                          # a human edit can plausibly move it

action_score = severity x log1p(imp_mar) x (1 + underclick) x (1 + 0.5 x reachable)
```

The gate is `0.8` — the bucket edge where 1c's decline rate crosses the base rate — and the two
multipliers (2x for underclicking, 1.5x for reachable) are hand-chosen, in that order, because
1b's gap (~+19pp) is roughly double what "reachable" is worth on its own. They are round numbers
picked by a human on purpose: this is the rule a person can argue with, not a fitted model.

Every page gets **exactly one** reason code and **one** action:

| Reason code | Fires when | Action for the editor |
|---|---|---|
| `slipping_and_underclicking` | sliding, in reach, below its band's median CTR | `rewrite_title_and_refresh` |
| `slipping_reachable_page` | sliding, in reach, CTR is fine for its band | `refresh_now` |
| `slipping_deep_page` | sliding, but ranked past 20 (or no usable position) | `expand_or_consolidate` |
| `holding_steady` | not sliding (severity 0) | `monitor` |

The cell below defines the rule once, as a function, and demonstrates it on four hand-made rows —
one per reason code — so the logic is readable before it touches 85,453 real pages.

In [7]:
SLIP_GATE   = 0.8     # momentum at/above this = not sliding (1c: the bucket edge at the base rate)
POS_REACH   = 20      # 1b: the band the shipped CTR-fix logic considers reachable
W_UNDERCLICK = 1.0    # doubles the score  (1 + 1.0)
W_REACHABLE  = 0.5    # 1.5x the score     (1 + 0.5)

ACTIONS = {
    "slipping_and_underclicking": "rewrite_title_and_refresh",
    "slipping_reachable_page":    "refresh_now",
    "slipping_deep_page":         "expand_or_consolidate",
    "holding_steady":             "monitor",
}
RULE_INPUTS = ["imp_mar", "momentum_in_month", "ctr_mar", "pos_mar"]   # nothing else may enter


def score_pages(df, band_med_ctr=None):
    """The whole baseline, in one readable function. Inputs: RULE_INPUTS only."""
    band_med_ctr = BAND_MED_CTR if band_med_ctr is None else band_med_ctr
    band = pd.cut(df.pos_mar, POS_BANDS, labels=POS_NAMES)

    severity   = (1 - df.momentum_in_month / SLIP_GATE).clip(lower=0)
    underclick = (df.ctr_mar < band.map(band_med_ctr).astype(float)).astype(int)
    reachable  = ((df.pos_mar >= 1) & (df.pos_mar <= POS_REACH)).astype(int)

    score = severity * np.log1p(df.imp_mar) * (1 + W_UNDERCLICK * underclick) * (1 + W_REACHABLE * reachable)

    reason = np.where(severity == 0, "holding_steady",
             np.where((underclick == 1) & (reachable == 1), "slipping_and_underclicking",
             np.where(reachable == 1, "slipping_reachable_page", "slipping_deep_page")))

    return pd.DataFrame({"severity": severity.round(3), "underclick": underclick,
                         "reachable": reachable, "action_score": score.round(3),
                         "reason_code": reason,
                         "action": pd.Series(reason, index=df.index).map(ACTIONS)}, index=df.index)


demo = pd.DataFrame({
    "imp_mar":           [20000,   20000,   20000,   20000],
    "momentum_in_month": [ 0.10,    0.10,    0.10,    1.05],
    "ctr_mar":           [0.0002, 0.0090, 0.0002, 0.0002],   # 0.02%, 0.90%, 0.02%, 0.02%
    "pos_mar":           [  6.0,     6.0,    41.0,     6.0],
})
print("the rule on four hand-made pages — one per reason code:")
print(pd.concat([demo, score_pages(demo)], axis=1).to_string(index=False))

assert set(score_pages(demo).reason_code) == set(ACTIONS), "a reason code is unreachable"
assert score_pages(demo).action.notna().all(), "every reason code needs exactly one action"
print("\nall four reason codes are reachable, and each maps to exactly one action.")

the rule on four hand-made pages — one per reason code:
 imp_mar  momentum_in_month  ctr_mar  pos_mar  severity  underclick  reachable  action_score                reason_code                    action
   20000               0.10   0.0002      6.0     0.875           1          1        25.997 slipping_and_underclicking rewrite_title_and_refresh
   20000               0.10   0.0090      6.0     0.875           0          1        12.998    slipping_reachable_page               refresh_now
   20000               0.10   0.0002     41.0     0.875           0          0         8.666         slipping_deep_page     expand_or_consolidate
   20000               1.05   0.0002      6.0     0.000           1          1         0.000             holding_steady                   monitor

all four reason codes are reachable, and each maps to exactly one action.


## 2. Build the ranked queue (writes the CSV)

The rule is applied to all 85,453 pages — the editor's queue is the whole slice, ranked — and
written to `work/outputs/baseline_action_score.csv`. That file is regenerated on every run.

The band-median CTR yardstick was learned on training clients in 1b and is applied unchanged to
every page, including the held-out ones.

In [8]:
scored = pd.concat([frame, score_pages(frame)], axis=1)

# every page gets exactly one reason code, and every reason code has an action
assert scored.reason_code.notna().all() and scored.action.notna().all()
assert scored.groupby("reason_code").action.nunique().eq(1).all(), "a reason code maps to two actions"

mix = (scored.groupby("reason_code")
             .agg(pages=("label_declining", "size"),
                  share=("label_declining", lambda s: len(s) / len(scored)),
                  median_score=("action_score", "median"),
                  decline_rate=("label_declining", "mean"))
             .sort_values("median_score", ascending=False))
print("the queue by reason code (whole slice)")
print(mix.round(4).to_string())
print(f"\npages with a score above zero (i.e. actually in the work queue): "
      f"{int((scored.action_score > 0).sum()):,} of {len(scored):,} "
      f"({(scored.action_score > 0).mean():.1%}) — the other {(scored.action_score == 0).mean():.0%} are 'monitor'.")

queue = (scored.sort_values("action_score", ascending=False)
               .assign(rank=lambda d: np.arange(1, len(d) + 1))
               [["rank", "client_hash_id", "content_hash_id", "action_score", "reason_code", "action",
                 "severity", "underclick", "reachable", "imp_mar", "pos_mar", "ctr_mar",
                 "momentum_in_month", "active_days", "spike_share", "split"]])

out_dir = pathlib.Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "baseline_action_score.csv"
queue.to_csv(csv_path, index=False)
print(f"\nranked queue written to work/outputs/{csv_path.name}: {len(queue):,} rows, "
      f"{queue.shape[1]} columns (regenerated on every run; gitignored by design)")
print("\nhead of the queue — ids withheld from the printout, they are pseudonyms and live only in the CSV:")
print(queue.head(5)[["rank", "action_score", "reason_code", "action", "imp_mar",
                     "momentum_in_month", "pos_mar", "ctr_mar"]].round(4).to_string(index=False))

receipts["queue_rows"] = int(len(queue))
receipts["queue_actionable_rows"] = int((scored.action_score > 0).sum())
receipts["reason_code_mix"] = {k: int(v) for k, v in mix.pages.items()}

the queue by reason code (whole slice)


                            pages   share  median_score  decline_rate
reason_code                                                          
slipping_and_underclicking  12292  0.1438        5.9475        0.7891
slipping_reachable_page      8357  0.0978        2.3780        0.6799
slipping_deep_page           7136  0.0835        1.7620        0.7200
holding_steady              57668  0.6749        0.0000        0.4411

pages with a score above zero (i.e. actually in the work queue): 27,785 of 85,453 (32.5%) — the other 67% are 'monitor'.



ranked queue written to work/outputs/baseline_action_score.csv: 85,453 rows, 16 columns (regenerated on every run; gitignored by design)

head of the queue — ids withheld from the printout, they are pseudonyms and live only in the CSV:
 rank  action_score                reason_code                    action  imp_mar  momentum_in_month  pos_mar  ctr_mar
    1        29.999 slipping_and_underclicking rewrite_title_and_refresh  58278.0             0.0710   8.3314   0.0001
    2        27.854 slipping_and_underclicking rewrite_title_and_refresh  65330.0             0.1301   4.6215   0.0005
    3        27.515 slipping_and_underclicking rewrite_title_and_refresh  64935.0             0.1379   7.6132   0.0003
    4        26.219 slipping_and_underclicking rewrite_title_and_refresh  57720.0             0.1623   6.0193   0.0007
    5        26.109 slipping_and_underclicking rewrite_title_and_refresh   9370.0             0.0387   4.3335   0.0001


### 2b. Does it beat picking at random — and does it beat the rules it replaces?

The lane's metric, fixed since w01, is **precision@50**: of the 50 pages the score puts at the
top, how many actually declined next month? Alone it means nothing, so it is read against three
things — the base rate (what an editor picking 50 pages blindly would get), the shipped rule
shaped like FlyRank's CTR-fix flag, and each single signal on its own.

Everything below is measured on the **held-out clients only** (9 clients, 25,771 pages).

I also report what each queue would actually have *saved*: the March→April impressions lost by
the 50 pages it picked. Precision counts pages; an editor's time is spent to protect traffic, and
50 correct picks on tiny pages protect very little of it.

In [9]:
te = scored[scored.split == "test"].copy()
te["lost_impressions"] = (te.imp_mar - te.imp_apr).clip(lower=0)   # label-side, reporting only
te["steepest_slide"] = -te.momentum_in_month
te["lowest_ctr"] = -te.ctr_mar
te["ctr_fix_flag_score"] = (((te.imp_mar >= 500) & (te.pos_mar >= 1) & (te.pos_mar <= POS_REACH)
                             & (te.ctr_mar * 100 < 0.5)).astype(int) * te.log_imp_mar)

rows = []
for name, col in [("my rule (baseline_action_score)", "action_score"),
                  ("steepest slide only", "steepest_slide"),
                  ("lowest CTR only", "lowest_ctr"),
                  ("biggest pages first", "log_imp_mar"),
                  ("product-shaped CTR-fix flag", "ctr_fix_flag_score")]:
    q = te.nlargest(TOP_K, col)
    rows.append({"rule": name, f"precision@{TOP_K}": round(q.label_declining.mean(), 3),
                 "median_imp_mar": int(q.imp_mar.median()),
                 "impressions_lost_by_queue": int(q.lost_impressions.sum()),
                 "clients_in_queue": int(q.client_hash_id.nunique())})

rng = np.random.default_rng(SEED)
draws = [te.iloc[rng.choice(len(te), TOP_K, replace=False)] for _ in range(200)]
rows.append({"rule": "picking 50 at random (200 draws)",
             f"precision@{TOP_K}": round(float(np.mean([d.label_declining.mean() for d in draws])), 3),
             "median_imp_mar": int(np.median([d.imp_mar.median() for d in draws])),
             "impressions_lost_by_queue": int(np.mean([d.lost_impressions.sum() for d in draws])),
             "clients_in_queue": int(np.median([d.client_hash_id.nunique() for d in draws]))})

comparison = pd.DataFrame(rows)
print(f"held-out clients: {len(te):,} pages | base rate {te.label_declining.mean():.2%}")
print(comparison.to_string(index=False))

p95_random = float(np.percentile([d.label_declining.mean() for d in draws], 95))
print(f"\nrandom picking: mean {np.mean([d.label_declining.mean() for d in draws]):.1%}, "
      f"95th percentile {p95_random:.1%} — a rule has to clear that to mean anything.")

curve = {k: round(precision_at_k(te, "action_score", k=k), 3) for k in (10, 25, 50, 100, 250)}
print(f"\nprecision@K for my rule as the queue gets longer: {curve}")

receipts["baseline_precision_at_50_test"] = round(precision_at_k(te, "action_score"), 4)
receipts["baseline_precision_curve_test"] = curve
receipts["random_top50_mean"] = round(float(np.mean([d.label_declining.mean() for d in draws])), 4)
receipts["random_top50_p95"] = round(p95_random, 4)
receipts["comparison_test"] = {r["rule"]: r[f"precision@{TOP_K}"] for r in rows}
receipts["impressions_lost_by_queue"] = {r["rule"]: r["impressions_lost_by_queue"] for r in rows}

held-out clients: 25,771 pages | base rate 49.36%
                            rule  precision@50  median_imp_mar  impressions_lost_by_queue  clients_in_queue
 my rule (baseline_action_score)         0.900            5698                     320709                 4
             steepest slide only         0.920             402                     117462                 5
                 lowest CTR only         0.800             225                       9759                 1
             biggest pages first         0.180           87275                     466773                 2
     product-shaped CTR-fix flag         0.160           71676                     363261                 2
picking 50 at random (200 draws)         0.502             742                      27540                 4

random picking: mean 50.2%, 95th percentile 62.0% — a rule has to clear that to mean anything.

precision@K for my rule as the queue gets longer: {10: 0.8, 25: 0.88, 50: 0.9, 100: 0.91, 250: 0.

**Reading the table honestly.**

- **My rule: precision@50 = 90.0%** on held-out clients, against a **49.4% base rate** and a
  random-draw 95th percentile of 62.0%. That is 45 of 50 picks correct where blind picking gets
  ~25 — a wide, defensible margin, and stable as the queue lengthens (80% @10, 90% @50, 91% @250),
  which matters because an editor's capacity is not exactly 50.
- **The rule it replaces does badly here.** The product-shaped CTR-fix flag scores **16.0%** —
  *below* random. It is not a broken idea; it is a rule pointed at a different question (where is
  there click opportunity) being graded on mine (what is about to bleed), with a threshold that
  fires on 82.5% of its pool so `log(impressions)` ends up doing the ranking. "Biggest pages
  first" (18.0%) shows the same thing: the largest pages are the *least* likely to drop 20%.
- **A single signal nearly matches me.** "Steepest slide only" scores
  **92.0%** — two pages better than my composite. But its 50 pages have a median of 402 March
  impressions and together lost **117,462** impressions, while my 50 have a median of 5,698 and
  lost **320,709** — 2.7x more traffic protected for one extra miss. Precision@50 counts pages,
  not traffic; I keep the size factor deliberately, and I report both numbers rather than the one
  that flatters the rule.

**The baseline is now frozen.** `precision@50 = 90.0%` on these held-out clients is the number
next week's model must beat — not the rule, not the thresholds, not the split. Moving any of them
after seeing a model score would make the comparison meaningless.

## 3. Top-10 review

The queue is only worth what its head is worth, so here are the ten pages the rule puts in front
of the editor first — each with its action, why it is there, and **what would make it wrong**.

The "what would make it wrong" line is built from decision-time information only (spike
concentration, month coverage, page size, position sanity) — never from what April actually did.
A reviewer on 2026-03-31 could raise every one of these objections; that is the point of the
column. Ids stay out of the printout; they are pseudonyms and live only in the CSV.

In [10]:
def why_it_is_there(r):
    band = pd.cut([r.pos_mar], POS_BANDS, labels=POS_NAMES)[0]
    med = BAND_MED_CTR.get(band, np.nan) * 100
    return (f"{r.reason_code}: {int(r.imp_mar):,} March impressions, second half ran "
            f"{r.momentum_in_month:.2f}x the first (severity {r.severity:.2f}), "
            f"CTR {r.ctr_mar * 100:.2f}% vs {med:.2f}% median at position {r.pos_mar:.1f}")


def what_would_make_it_wrong(r):
    if r.spike_share >= 0.25:
        return (f"one March day carries {r.spike_share:.0%} of its impressions — the 'slide' may be "
                f"that spike leaving the window, not the page decaying")
    if r.active_days < 25:
        return (f"live on only {int(r.active_days)} of 31 March days — partial coverage can fake a "
                f"second-half slide")
    if r.imp_mar < 500:
        return (f"small page ({int(r.imp_mar):,} impressions) — a 20% drop here costs little, so a "
                f"correct pick is still a poor use of the editor's slot")
    if r.pos_mar < 1:
        return "recorded average position below 1 is not a real rank — the position input is unreliable here"
    if r.ctr_mar * 100 < 0.05:
        return ("CTR is near zero despite a good rank — the page may be ranking for queries it cannot "
                "answer, and a title rewrite would not fix an intent mismatch")
    return ("a March slide can continue or bounce back — the rule cannot tell a seasonal dip from "
            "genuine decay, and April is one month")


top10 = scored.nlargest(10, "action_score").reset_index(drop=True)
for i, r in top10.iterrows():
    print(f"#{i + 1:<2} {r.action:<26} score {r.action_score:6.2f}   [{r.split} client]")
    print(f"    why : {why_it_is_there(r)}")
    print(f"    risk: {what_would_make_it_wrong(r)}")

print(f"\ndistinct clients in the top 10: {top10.client_hash_id.nunique()} "
      f"(of {scored.client_hash_id.nunique()} in the slice)")
print(f"actions issued: {top10.action.value_counts().to_dict()}")
print(f"how the ten actually did in April: {int(top10.label_declining.sum())}/10 declined "
      f"(read after the review, not before)")

receipts["top10_hit_rate"] = round(float(top10.label_declining.mean()), 4)
receipts["top10_distinct_clients"] = int(top10.client_hash_id.nunique())
receipts["top10_spike_flagged"] = int((top10.spike_share >= 0.25).sum())

#1  rewrite_title_and_refresh  score  30.00   [train client]
    why : slipping_and_underclicking: 58,278 March impressions, second half ran 0.07x the first (severity 0.91), CTR 0.01% vs 0.19% median at position 8.3
    risk: one March day carries 64% of its impressions — the 'slide' may be that spike leaving the window, not the page decaying
#2  rewrite_title_and_refresh  score  27.85   [train client]
    why : slipping_and_underclicking: 65,330 March impressions, second half ran 0.13x the first (severity 0.84), CTR 0.05% vs 0.19% median at position 4.6
    risk: one March day carries 37% of its impressions — the 'slide' may be that spike leaving the window, not the page decaying
#3  rewrite_title_and_refresh  score  27.52   [train client]
    why : slipping_and_underclicking: 64,935 March impressions, second half ran 0.14x the first (severity 0.83), CTR 0.03% vs 0.19% median at position 7.6
    risk: CTR is near zero despite a good rank — the page may be ranking for queries it cannot

**What the ten say when read together.**

- **They are all one reason code.** Every one is `slipping_and_underclicking` →
  `rewrite_title_and_refresh`. That is the 2x multiplier doing exactly what it was designed to do,
  but it means the head of the queue offers an editor no variety of work — no `refresh_now` or
  `expand_or_consolidate` page appears anywhere in the top 20. A queue that only ever asks for one kind of
  task is a queue that will get one kind of attention.
- **They come from very few clients** — 4 of 27. The slice is concentrated (w03 measured the top 3
  clients at 60.5% of rows), and the rule inherits that concentration without correcting for it.
  An editor working this queue would spend a day inside three or four accounts.
- **Half of them carry a spike warning.** Five of the ten have a single March day holding 25%+ of
  the month's impressions — for those, "the second half is quieter" may just be the spike leaving
  the window. The rule cannot see the difference; the reviewer can, which is why the line is
  printed next to the action.
- **Read after the review: 8 of the 10 declined.** Two did not, and one of those actually grew.
  That is the honest shape of a rule whose top picks look extreme: it finds pages in motion, and
  motion sometimes reverses.

## 4. Weak picks + leakage check

Three things a reviewer would ask: which picks look wrong on the evidence available at the
decision moment, what the rule is *blind* to, and whether anything from after the decision moment
crept into the score.

In [11]:
top20 = scored.nlargest(20, "action_score")
print("--- weak picks in the top 20, judged on decision-time evidence only ---")
weak = pd.DataFrame({
    "one day holds >=25% of March impressions": [int((top20.spike_share >= 0.25).sum())],
    "live fewer than 25 of 31 March days":      [int((top20.active_days < 25).sum())],
    "small page (<500 March impressions)":      [int((top20.imp_mar < 500).sum())],
    "position below 1 (unusable rank)":         [int((top20.pos_mar < 1).sum())],
}).T.rename(columns={0: "pages_of_20"})
print(weak.to_string())
print(f"\nmedian spike share in the top 20: {top20.spike_share.median():.0%} "
      f"vs {scored.spike_share.median():.0%} across the whole slice — the head of the queue is "
      f"visibly spikier than the population it came from.")

print("\n--- the blind spot: what the rule never shows the editor ---")
zeros = te[te.action_score == 0]
missed = int(zeros.label_declining.sum())
print(f"held-out pages scored 0 (never queued): {len(zeros):,}")
print(f"of the {int(te.label_declining.sum()):,} pages that actually declined in April, "
      f"{missed:,} ({missed / te.label_declining.sum():.1%}) were NOT sliding inside March — "
      f"the rule cannot see them at all.")
print("That is the gap a model has to close: declines that start without a within-month warning.")

print("\n--- leakage check: could anything from April have reached the score? ---")
print(f"1. rule inputs: {RULE_INPUTS}")
assert set(RULE_INPUTS) <= set(CONTRACT_FEATURES) | {"imp_mar"}, "a rule input is outside the w03 contract"
print("   all of them are w03 contract features (imp_mar is log_imp_mar's source column).")

LABEL_SIDE = ["imp_apr", "label_declining", "lost_impressions"]
blind_frame = frame.drop(columns=[c for c in LABEL_SIDE if c in frame.columns])
assert not any(c in blind_frame.columns for c in LABEL_SIDE)
rescored = score_pages(blind_frame)
assert rescored.action_score.equals(scored.action_score), "the score changed without the label columns"
print("2. re-scored a frame with every label column physically dropped: identical scores "
      "(so the score cannot be reading the label, even by accident).")

feature_month_only = con.sql(f"""
    SELECT COUNT(*) AS rows_outside_march
    FROM {fact(FEATURE_MONTH)}
    WHERE report_date < DATE '{FEATURE_MONTH}-01' OR report_date > DATE '{DECISION_DATE}'
""").df()
print(f"3. every feature row comes from {FEATURE_MONTH}; rows outside the feature window in that "
      f"partition: {int(feature_month_only.rows_outside_march[0])}.")

print("4. the band-median CTR yardstick was computed on training clients only "
      f"({int((frame.split == 'train').sum()):,} rows) and applied unchanged to held-out pages.")
print("5. no product flag (health_score, priority_score, action_type, refresh flags) is an input — "
      "the CTR-fix flag appears only as a baseline to beat in section 2b.")

receipts["blind_spot_missed_declines"] = missed
receipts["blind_spot_missed_share"] = round(float(missed / te.label_declining.sum()), 4)
receipts["top20_spike_flagged"] = int((top20.spike_share >= 0.25).sum())
receipts["frozen"] = True

out_path = out_dir / "w04_baseline_score_receipts.json"
out_path.write_text(json.dumps(receipts, indent=2), encoding="utf-8")
print(f"\nreceipts written to work/outputs/{out_path.name} ({len(receipts)} entries)")
print(json.dumps(receipts, indent=2))

--- weak picks in the top 20, judged on decision-time evidence only ---
                                          pages_of_20
one day holds >=25% of March impressions           11
live fewer than 25 of 31 March days                 0
small page (<500 March impressions)                 0
position below 1 (unusable rank)                    0

median spike share in the top 20: 33% vs 8% across the whole slice — the head of the queue is visibly spikier than the population it came from.

--- the blind spot: what the rule never shows the editor ---
held-out pages scored 0 (never queued): 16,556
of the 12,721 pages that actually declined in April, 6,746 (53.0%) were NOT sliding inside March — the rule cannot see them at all.
That is the gap a model has to close: declines that start without a within-month warning.

--- leakage check: could anything from April have reached the score? ---
1. rule inputs: ['imp_mar', 'momentum_in_month', 'ctr_mar', 'pos_mar']
   all of them are w03 contract featu

3. every feature row comes from 2026-03; rows outside the feature window in that partition: 0.
4. the band-median CTR yardstick was computed on training clients only (59,682 rows) and applied unchanged to held-out pages.
5. no product flag (health_score, priority_score, action_type, refresh flags) is an input — the CTR-fix flag appears only as a baseline to beat in section 2b.

receipts written to work/outputs/w04_baseline_score_receipts.json (38 entries)
{
  "lane": "bleed_tracker",
  "seed": 42,
  "top_k": 50,
  "feature_month": "2026-03",
  "label_month": "2026-04",
  "slice_rows": 85453,
  "slice_clients": 27,
  "slice_base_rate": 0.5378,
  "train_rows": 59682,
  "test_rows": 25771,
  "test_clients": 9,
  "test_base_rate": 0.4936,
  "s1a_updated_after_decision_share": 0.8608,
  "s1a_testable_pages": 8305,
  "s1a_flag_pool": 4,
  "s1a_verdict": "FALSE",
  "s1b_band_median_ctr_pct": {
    "1-3": 0.1775,
    "4-10": 0.1889,
    "11-20": 0.1165,
    "21+": 0.0
  },
  "s1b_underclick_ga

**What I would tell the team.**

The baseline works, and it works for a narrow reason: **it finds pages already in motion.** On
held-out clients it puts 45 correct pages in an editor's first 50 against ~25 for blind picking,
and the traffic those 50 pages lost is 2.7x what the best single-signal rule would have caught.
As a decision-support queue it is worth shipping today.

Its three honest limits, in the order they would bite:

1. **It is a persistence rule, not a prediction.** The label compares April to all of March, so a
   page that collapsed in the second half of March is partly guaranteed to score as declining.
   The 90% is measured, and it is measuring continuation. The number that would separate the two
   is precision on pages that were *flat* in March — the 53.0% of held-out declines the rule
   currently cannot see at all.
2. **It is one month pair, and a concentrated one.** March→April 2026, nine held-out clients, a
   top-10 drawn from three of them. Anything that moved search broadly in April 2026 is inside
   this number. Repeating the contract on other month pairs is what would turn a
   directional result into a stable one.
3. **The head of the queue is spikier than the population** — 11 of the top 20 have a single
   March day carrying a quarter or more of the month's impressions. Before the queue reaches a
   human, a spike guard (require, say, the top day to hold under 30%, or measure the slide on
   medians instead of sums) is the first change I would make. I am leaving it out of *this*
   notebook on purpose: the baseline is frozen, and improving it after seeing its score is how
   baselines quietly stop being baselines.

**Two things this week changed for the lane.** Staleness is out — not deprioritised, *unusable*
in this release (1a), and the fix is a data request, not a modelling choice. And the CTR-fix
threshold is out while its idea is in: relative CTR within a position band earns its place, the
0.5% constant does not (1b).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.